# Qwen 2.5 14B BF16 FineTuning

In [ ]:
import os
os.environ["HF_TOKEN"]      = "####"
os.environ["WANDB_API_KEY"] = "####"

In [ ]:
# Redirect HF cache to scratch partition — must happen before any HF import
os.environ["HF_HOME"]            = "/tmp/mawojide/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/mawojide/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/mawojide/hf_cache/datasets"
os.environ["HF_TEMP_DIR"]        = "/tmp/huggingface_temp"

os.makedirs("/tmp/mawojide/hf_cache/transformers", exist_ok=True)
os.makedirs("/tmp/mawojide/hf_cache/datasets",     exist_ok=True)

print("HF_HOME →", os.environ["HF_HOME"])

from huggingface_hub import constants
print("HF_HUB_CACHE →", constants.HF_HUB_CACHE)

In [ ]:
# Install dependencies
! pip install "transformers>=4.46" trl peft accelerate datasets huggingface_hub -q
! pip install bitsandbytes>=0.43 scikit-learn plotly tqdm -q
! pip install wandb

In [ ]:
import os, re, json, math, statistics, gc
from datetime import datetime
from itertools import accumulate
from IPython.display import clear_output

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import login, HfApi

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8   # True on H100 (sm_90)
print(f"bfloat16       : {USE_BF16}  (compute capability {capability})")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM           : {vram_gb:.1f} GB")

In [ ]:
# ── Confirm clean GPU state before anything is loaded ──────────────────────
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reserved  : {torch.cuda.memory_reserved()  / 1e9:.2f} GB")

## Constants & Hyperparameters

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
PROJECT_NAME = "price"
HF_USER      = "martinsawojide"

# ── Dataset ────────────────────────────────────────────────────────────────
LITE_MODE         = False
DATA_USER         = "ed-donner"
DATASET_NAME      = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

dataset = load_dataset(DATASET_NAME)
train   = dataset["train"]
val     = dataset["val"]
test    = dataset["test"]
print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

# ── Run naming ─────────────────────────────────────────────────────────────
RUN_NAME         = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" + ("-lite" if LITE_MODE else "")
PROJECT_RUN_NAME = f"{PROJECT_NAME}-qwen2.5-14b-{RUN_NAME}"
HUB_MODEL_NAME   = f"{HF_USER}/{PROJECT_RUN_NAME}"
CHECKPOINT_DIR   = f"/tmp/checkpoint_dir/{PROJECT_RUN_NAME}"

# ── Sequence length ────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 1024

# ── QLoRA config ───────────────────────────────────────────────────────────
LORA_R          = 32      
LORA_ALPHA      = 64      
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS          = 5   if LITE_MODE else 1
BATCH_SIZE      = 4
GRAD_ACCUM      = 4   if LITE_MODE else 8
LEARNING_RATE   = 2e-4   # QLoRA benefits from higher LR than full fine-tuning
WARMUP_STEPS    = 750  if LITE_MODE else 50
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.01
OPTIMIZER       = "paged_adamw_8bit"   # bitsandbytes paged optimizer — saves GPU memory
MAX_GRAD_NORM   = 1.0

# ── Logging / saving ───────────────────────────────────────────────────────
LOG_STEPS    = 5    if LITE_MODE else 10
SAVE_STEPS   = 50   if LITE_MODE else 100
VAL_SIZE     = len(val)
LOG_TO_WANDB = True

if LOG_TO_WANDB:
    import wandb
    os.environ["WANDB_API_KEY"]   = os.getenv("WANDB_API_KEY", "")
    os.environ["WANDB_PROJECT"]   = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"]     = "false"
    wandb.login()

# ── Inference ──────────────────────────────────────────────────────────────
BEAM_NUM_BEAMS   = 10
BEAM_NUM_GROUPS  = 10
BEAM_DIV_PENALTY = 1.5
EVAL_SIZE        = len(test)

print(f"Run name       : {RUN_NAME}")
print(f"Hub model      : {HUB_MODEL_NAME}")
print(f"Checkpoint dir : {CHECKPOINT_DIR}")
print(f"Dataset        : {DATASET_NAME}")
print(f"Max seq length : {MAX_SEQ_LENGTH}")
print(f"LoRA rank/alpha: {LORA_R} / {LORA_ALPHA}")
print(f"Batch / accum  : {BATCH_SIZE} / {GRAD_ACCUM}  (effective {BATCH_SIZE * GRAD_ACCUM})")
print(f"Optimizer      : {OPTIMIZER}")
print(f"Epochs         : {EPOCHS}")

## 3 · HuggingFace Login

In [ ]:
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

api = HfApi()
try:
    user = api.whoami()
    print(f"Logged in as : {user['name']}")
    print(f"Token role   : {user['auth']['accessToken']['role']}")
except Exception as e:
    print(f"Login check failed: {e}")

## 4 · Load & Inspect Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
train   = dataset["train"]
val     = dataset["val"].select(range(VAL_SIZE))
test    = dataset["test"]

print(f"Train : {len(train):,} rows")
print(f"Val   : {len(val):,} rows")
print(f"Test  : {len(test):,} rows")
print(f"\nColumns : {train.column_names}")
print(f"\nSample prompt   : {train[0]['prompt'][:200]}")
print(f"Sample completion: {train[0]['completion']}")

## Load Base Model in BF16

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # SFTTrainer expects right-padding for packing

print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"EOS token      : '{tokenizer.eos_token}'  (id {tokenizer.eos_token_id})")
print(f"PAD token      : '{tokenizer.pad_token}'  (id {tokenizer.pad_token_id})")
print(f"Chat template  : {'present' if tokenizer.chat_template else 'MISSING — check transformers version'}")

In [ ]:
# Load base model for evaluation only — no training overhead
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype        = torch.bfloat16 if USE_BF16 else torch.float16,
    device_map          = "cuda",
    attn_implementation = "flash_attention_2",  # H100 supports FA2 natively
)
base_model.eval()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded   : {BASE_MODEL}")
print(f"VRAM used      : {vram_used:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## Inference Helpers

In [ ]:
def build_prompt(item: dict, include_completion: bool = False) -> str:
    """
    Wrap a dataset item in Llama 3.3 chat format using apply_chat_template.

    If include_completion=True, appends the price as the assistant turn
    (used when building SFT training sequences).
    If False, leaves the assistant turn open (used for inference).
    """
    messages = [{"role": "user", "content": str(item["prompt"])}]

    if include_completion:
        messages.append({
            "role": "assistant",
            "content": str(item["completion"])
        })
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


# Verify chat template wrapping on a sample
sample_text = build_prompt(train[0])
print("Llama-formatted inference prompt:")
print(sample_text[:400])
print("...")
token_count = len(tokenizer.encode(sample_text))
print(f"\nToken count (no completion): {token_count}")

In [ ]:
def extract_price(text: str):
    """Extract the first plausible USD price from generated text."""
    patterns = [
        r"\$[\d,]+\.?\d*",   # $12.99  $1,299
        r"[\d,]+\.\d{2}",    # 12.99
        r"[\d,]+",              # 12
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace("$", "").replace(",", "")
            try:
                price = float(price_str)
                if 0.01 <= price <= 100_000:
                    return price
            except ValueError:
                continue
    return None


def predict_with_model(model, tok, item,
                       num_beams=BEAM_NUM_BEAMS,
                       num_groups=BEAM_NUM_GROUPS,
                       diversity_penalty=BEAM_DIV_PENALTY):
    """
    Diverse beam search → geometric mean of all beam prices.
    """
    prompt       = build_prompt(item, include_completion=False)
    inputs       = tok(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens       = 256,
            num_beams            = num_beams,
            num_beam_groups      = num_groups,
            diversity_penalty    = diversity_penalty,
            num_return_sequences = num_beams,
            pad_token_id         = tok.eos_token_id,
            use_cache            = True,
            trust_remote_code       = True,
            do_sample             = False,  # greedy decoding within each beam group
        )

    prices = []
    for beam_output in output_ids:
        text  = tok.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    if not prices:
        return None
    return statistics.geometric_mean(prices)


def base_model_predict(item):
    return predict_with_model(base_model, tokenizer, item)

## Evaluation Framework (Tester + Plotly Charts)

In [ ]:
GREEN     = "\033[92m"
YELLOW    = "\033[93m"
RED       = "\033[91m"
RESET     = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


class Tester:
    def __init__(self, predictor, data, title=None, size=EVAL_SIZE):
        self.predictor = predictor
        self.data      = data
        self.title     = title or self.make_title(predictor)
        self.size      = size
        self.titles    = []
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.colors    = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title()

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        return value if value is not None else 0

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:    return "green"
        elif error < 80 or error / truth < 0.4:  return "orange"
        else:                                     return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value     = self.predictor(datapoint)
        guess     = self.post_process(value)
        truth     = float(datapoint["completion"])
        error     = abs(guess - truth)
        color     = self.color_for(error, truth)
        pieces    = datapoint["prompt"].split("Title: ")
        title     = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title     = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses,
            "title": self.titles, "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=800, height=600,
        )
        for tr in fig.data:
            mask            = df["color"] == tr.name
            tr.customdata   = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n               = len(self.errors)
        running_sums    = list(accumulate(self.errors))
        x               = list(range(1, n + 1))
        running_means   = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds    = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci    = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        title = f"{self.title}  |  Final MAE: ${running_means[-1]:,.2f} ± ${ci[-1]:,.2f}"
        fig   = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines",
            line=dict(width=3, color="firebrick"),
            customdata=list(zip(ci)),
            hovertemplate="n=%{x}<br>Avg Error=$%{y:,.2f}<br>±95% CI=$%{customdata[0]:,.2f}<extra></extra>",
        ))
        fig.update_layout(
            title=title, xaxis_title="Number of Datapoints",
            yaxis_title="MAE ($)", width=800, height=300,
            template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        avg_error = sum(self.errors) / self.size
        mse       = mean_squared_error(self.truths, self.guesses)
        r2        = r2_score(self.truths, self.guesses) * 100
        title     = (f"{self.title}<br>"
                     f"<b>MAE:</b> ${avg_error:,.2f}  "
                     f"<b>MSE:</b> {mse:,.0f}  "
                     f"<b>r²:</b> {r2:.1f}%")
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size), desc=self.title):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        clear_output(wait=True)
        self.report()
        return self


def evaluate(function, data, size=EVAL_SIZE):
    return Tester(function, data, size=size).run()

## Base Model Evaluation

In [ ]:
set_seed(42)
base_tester = evaluate(base_model_predict, test, size=EVAL_SIZE)

## Prepare Dataset for SFT

In [ ]:
def format_for_sft(examples):
    """
    Build full Llama chat-formatted training sequences.
    Each sequence: [Llama user turn][Llama assistant turn with price][EOS]
    """
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        item = {"prompt": prompt, "completion": completion}
        full_text = build_prompt(item, include_completion=True)
        if not full_text.endswith(tokenizer.eos_token):
            full_text += tokenizer.eos_token
        texts.append(full_text)
    return {"text": texts}


train_sft = train.map(format_for_sft, batched=True, remove_columns=train.column_names)
val_sft   = val.map(format_for_sft,   batched=True, remove_columns=val.column_names)

print("Sample SFT text:")
print(train_sft[0]["text"][:400])
sample_lengths = [len(tokenizer.encode(t)) for t in train_sft.select(range(200))["text"]]
print(f"\nToken length (200 samples) — min: {min(sample_lengths)}  max: {max(sample_lengths)}  mean: {sum(sample_lengths)/len(sample_lengths):.0f}")
print(f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} covers {sum(l<=MAX_SEQ_LENGTH for l in sample_lengths)/len(sample_lengths)*100:.1f}% of samples")
print(f"\nTrain SFT size : {len(train_sft):,}")
print(f"Val SFT size   : {len(val_sft):,}")

## LoRA Fine-Tuning

In [ ]:
# Free base model VRAM before training run
del base_model
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM after cleanup : {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = LORA_TARGET_MODULES,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)

print(f"LoRA rank        : {LORA_R}")
print(f"LoRA alpha       : {LORA_ALPHA}")
print(f"Target modules   : {LORA_TARGET_MODULES}")

In [ ]:
training_args = SFTConfig(
    # ── Output ─────────────────────────────────────────────────────────────
    output_dir              = CHECKPOINT_DIR,

    # ── Training duration ──────────────────────────────────────────────────
    num_train_epochs        = EPOCHS,
    max_steps               = -1,

    # ── Batch & gradient ───────────────────────────────────────────────────
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # ── Optimiser & LR ─────────────────────────────────────────────────────
    optim               = OPTIMIZER,           
    learning_rate       = LEARNING_RATE,
    weight_decay        = WEIGHT_DECAY,
    warmup_steps        = WARMUP_STEPS,
    lr_scheduler_type   = LR_SCHEDULER,
    max_grad_norm       = MAX_GRAD_NORM,

    # ── Precision (H100) ───────────────────────────────────────────────────
    fp16                = False,
    bf16                = USE_BF16,

    # ── Sequence / packing ─────────────────────────────────────────────────
    max_length      = MAX_SEQ_LENGTH,
    packing             = True,
    dataset_text_field  = "text",

    # ── Logging ────────────────────────────────────────────────────────────
    logging_steps       = LOG_STEPS,
    report_to           = "wandb" if LOG_TO_WANDB else "none",
    run_name            = RUN_NAME,

    # ── Checkpointing ──────────────────────────────────────────────────────
    save_strategy       = "steps",
    save_steps          = SAVE_STEPS,
    save_total_limit    = 3,

    # ── Evaluation ─────────────────────────────────────────────────────────
    eval_strategy       = "steps",
    eval_steps          = SAVE_STEPS,

    # ── Hub push ───────────────────────────────────────────────────────────
    push_to_hub         = False,

    # ── Misc ───────────────────────────────────────────────────────────────
    seed                    = 3407,
    dataloader_num_workers  = 4,
    remove_unused_columns   = False,
)

print(f"Output dir     : {CHECKPOINT_DIR}")
print(f"Hub target     : {HUB_MODEL_NAME}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Load model fresh for training, then wrap with LoRA
train_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype        = torch.bfloat16 if USE_BF16 else torch.float16,
    device_map          = "cuda",
    attn_implementation = "flash_attention_2",
)

trainer = SFTTrainer(
    model         = train_model,
    processing_class     = tokenizer,
    train_dataset = train_sft,
    eval_dataset  = val_sft,
    peft_config   = lora_config,
    args          = training_args,
)

total_params    = sum(p.numel() for p in trainer.model.parameters())
trainable       = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable:,}  ({100 * trainable / total_params:.2f}% — LoRA only)")

### Start training

In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps    : {trainer_stats.global_step}")
print(f"Training loss  : {trainer_stats.training_loss:.4f}")
print(f"Runtime        : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

## Save LoRA Adapter

In [ ]:
# Save LoRA adapter weights only
trainer.model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Saved locally  → {CHECKPOINT_DIR}/")

# Push adapter to Hub (small — only ~500 MB)
trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Pushed to Hub  → {HUB_MODEL_NAME}")

## Load Fine-Tuned Model for Inference

In [ ]:
# Free training model
del trainer, train_model
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM after training cleanup : {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# LOAD_FROM = HUB_MODEL_NAME   # change to CHECKPOINT_DIR to load locally
LOAD_FROM = "martinsawojide/price-qwen2.5-14b-2026-03-08_07.21.49"

ft_tokenizer = AutoTokenizer.from_pretrained(LOAD_FROM)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"

base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype        = torch.bfloat16 if USE_BF16 else torch.float16,
    device_map          = "cuda",
    attn_implementation = "flash_attention_2",
)
fine_tuned_model = PeftModel.from_pretrained(base_for_merge, LOAD_FROM)
fine_tuned_model = fine_tuned_model.merge_and_unload()
fine_tuned_model.eval()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"Fine-tuned model loaded : {LOAD_FROM}")
print(f"VRAM used               : {vram_used:.1f} GB")

## Fine-Tuned Model Evaluation

In [ ]:
def fine_tuned_predict(item):
    """Inference wrapper using the fine-tuned model — same beam config as base eval."""
    return predict_with_model(fine_tuned_model, ft_tokenizer, item)

In [ ]:
set_seed(42)
finetune_tester = evaluate(fine_tuned_predict, test, size=EVAL_SIZE)